# Numbers quoted in the report

Reproduce the numbers of report section 3.2 from the results of
[02_02](02_02_global_oracle.ipynb) and [02_03](02_03_selector_sweep.ipynb).
The figure is in [02_04](02_04_analyze_oracle.ipynb).

In [1]:
import pandas as pd

## 1. Data

Latency is in ms. Retained importance is $100\sum_i v_iM_i/\sum_i v_i$.

In [2]:
reference = pd.read_csv("oracle_reference.csv", float_precision="round_trip")
selectors = pd.read_csv("selector_results.csv", float_precision="round_trip")

# Combine the recorded methods.
results = pd.concat([reference, selectors], ignore_index=True)

In [3]:
# importance
results["retained"] = 100 * results.importance / results.importance_total

## 2. Traces and breakpoints

Count the traces of each shape. `saturation_rows` is the two-line breakpoint
in rows, $\ell_0=208/w$ for rows of $w$ KiB.

In [4]:
shapes = results.groupby(["n", "d"]).agg(
    traces          = ("trace_id", "nunique"),
    saturation_rows = ("saturation_rows", "first"),
)

shapes.round(2)

traces  saturation_rows
n    d                            
896  128      144           832.00
     896      144           118.86
     4864     144            21.89
4864 896       72           118.86

In [5]:
shapes.traces.sum()

np.int64(504)

## 3. Ratio to Global DP

A mask's ratio is its importance per ms of latency. Divide each method's ratio
by the Global DP ratio of the same trace and requested fraction.

In [6]:
results["ratio"] = results.importance / results.latency

In [7]:
KEYS = ["trace_id", "fraction"]

optimum = results[results.method == "Global DP"]
optimum = optimum.set_index(KEYS).ratio.rename("dp_ratio")

In [8]:
results = results.join(
    optimum,
    on       = KEYS,
    validate = "many_to_one",
)

results["vs_dp"] = 100 * results.ratio / results.dp_ratio

## 4. Medians at a 50% budget

For each shape and method, compute every column's median across traces.

In [9]:
FRACTION = 0.5
COLUMNS  = ["R_nominal", "tile_width", "retained", "latency", "vs_dp"]

In [10]:
selected = results[results.fraction == FRACTION]
summary  = selected.groupby(["n", "d", "method"])[COLUMNS].median()

### 4.1 $896\times896$

`Tiles (1s)` uses $B=\lceil\ell_0\rceil$.

In [11]:
METHODS = ["Global DP", "Paper greedy", "Tiles (1s)", "Top-R"]
DIGITS  = {"retained": 2, "latency": 4, "vs_dp": 2}

In [12]:
summary.loc[(896, 896)].loc[METHODS].round(DIGITS)

,R_nominal,tile_width,retained,latency,vs_dp
method,,,,,
Global DP,448.0,NaN,58.44,0.1441,100.00
Paper greedy,448.0,NaN,58.97,0.1483,96.82
Tiles (1s),448.0,119.0,56.32,0.1470,95.87
Top-R,448.0,1.0,72.48,2.6400,7.22


### 4.2 All shapes

Ratio to Global DP in percent.

In [13]:
ratios = summary.vs_dp.unstack("method")

ratios[["Paper greedy", "Tiles (1s)", "Top-R"]].round(2)

method     Paper greedy  Tiles (1s)  Top-R
n    d                                    
896  128          99.72      100.00   1.30
     896          96.82       95.87   7.22
     4864         97.51       95.78  26.60
4864 896          98.68       96.46   6.10